# HomeReach Melbourne — 09: Rental forecasting baselines

**A research experiment, not a new live-app feature.** We evaluate two simple ways of predicting one quarter ahead for each LGA's published median weekly rent for **two-bedroom flats**. The target is an area statistic in nominal **AUD/week**, not a property's rent or a live listing.

The two methods are **last observation** (copy the preceding quarter) and **seasonal naive** (copy the same quarter one year earlier). The horizon is one quarter for **both** methods: the age of the value copied is not the forecast horizon.

Use the existing **homereach** kernel. Work on **`feat/rental-forecast-baselines`**, not the deployed `main` branch. This notebook adds only its own research outputs and report. It does not change your application, release package, registry, original sources, or earlier pipelines.

### Evaluation plan — set before looking at forecast errors

| Role | Quarters | Quarters per LGA | What happens here |
|---|---|---:|---|
| Initial history | 1999Q2–2018Q4 | 79 | Available before the first validation forecast |
| Rolling validation | 2019Q1–2023Q3 | 19 | Produce and score successive one-quarter-ahead forecasts |
| Reserved final evaluation | 2023Q4–2025Q3 | 8 | **No forecasts or error scores in this notebook** |

These are project design choices, not publisher-supplied training/test labels. All 31 LGAs use the same dates. At each validation target, earlier validation observations become available to later origins; future observations do not.

**Honesty about the holdout:** recent rents have already appeared in your descriptive charts and deployed explorer. They are reserved from **modelling evaluation from this point onward**, not a pristine dataset nobody has seen. The input is read for structural verification, but holdout outcomes do not enter baseline predictions, metrics, charts, or exported prediction tables.

**Honesty about historical availability:** this uses one downloaded workbook vintage. It cannot reconstruct past publication delays, revisions, historical boundary definitions, or the exact data available on a historical date. Results are **retrospective, period-based validation**, not real-time performance. We do not backfill October 2026 transport indicators into rental history.

Primary method references: [naive and seasonal-naive methods](https://otexts.com/fpp3/simple-methods.html), [rolling-origin evaluation](https://otexts.com/fpp3/tscv.html), and [forecast accuracy](https://otexts.com/fpp3/accuracy.html).

## 1. Environment, branch and protected inputs

The notebook reads your existing `data/processed/rent_quarterly.csv`, its quality report, the audited rental settings, and the new `config/rental_forecast_protocol.json`.

It does **not** read GTFS. No installations or downloads occur. The optional raw rental workbook and current deployment files are fingerprinted only to verify they remain unchanged. A branch check prevents accidentally saving this experiment on deployed `main`.

In [1]:
print("Starting notebook 09 — rental forecasting baselines...", flush=True)
from pathlib import Path
from datetime import datetime, timezone
from tempfile import TemporaryDirectory
import hashlib
import json
import math
import subprocess
import sys

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly
from IPython.display import display
from homereach.io.dffh_rent import COLUMN_ORDER, read_panel_csv


def require(condition, message):
    if not bool(condition):
        raise ValueError(message)


def sha256_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def read_json(path):
    value = json.loads(Path(path).read_text(encoding="utf-8-sig"))
    require(isinstance(value, dict), f"Expected a JSON object: {path}")
    return value


ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "pyproject.toml").is_file() and (p / "data").is_dir()), None)
require(ROOT is not None, "Open this notebook from the existing HomeReach project.")
branch_result = subprocess.run(
    ["git", "-C", str(ROOT), "branch", "--show-current"],
    capture_output=True, text=True, timeout=10, check=True,
)
BRANCH = branch_result.stdout.strip()
require(BRANCH == "feat/rental-forecast-baselines",
        "Use the research branch feat/rental-forecast-baselines before running notebook 09; do not develop on deployed main.")

INPUT = ROOT / "data/processed/rent_quarterly.csv"
QUALITY_PATH = ROOT / "data/processed/rent_quality_report.json"
SPEC_PATH = ROOT / "config/rent_snapshot.json"
PROTOCOL_PATH = ROOT / "config/rental_forecast_protocol.json"
for path in [INPUT, QUALITY_PATH, SPEC_PATH, PROTOCOL_PATH]:
    require(path.is_file(), f"Missing required file: {path.relative_to(ROOT)}")

spec = read_json(SPEC_PATH)
quality_report = read_json(QUALITY_PATH)
protocol = read_json(PROTOCOL_PATH)
EXPECTED_SOURCE_SHA256 = "b8e96347494f09fea2a2ace55997fd34c9398f8509cb3450d65444aebfaaa837"
require(spec["sha256"] == protocol["source_sha256"] == EXPECTED_SOURCE_SHA256,
        "This notebook requires the audited 2025Q3 rental snapshot. Do not change fingerprints to bypass it.")
protected_paths = [INPUT, QUALITY_PATH, SPEC_PATH, PROTOCOL_PATH]
raw_rent = ROOT / "data/raw/rent" / spec["filename"]
if raw_rent.is_file():
    require(sha256_file(raw_rent) == EXPECTED_SOURCE_SHA256, "The original rental workbook changed.")
    protected_paths.append(raw_rent)
for relative in ["data/processed/rent_quarterly.parquet", "docs/data_registry.csv", "README.md"]:
    if (ROOT / relative).is_file():
        protected_paths.append(ROOT / relative)
for directory in ["app", "deploy"]:
    protected_paths.extend(p for p in (ROOT / directory).rglob("*")
                           if p.is_file() and "__pycache__" not in p.parts and p.suffix != ".pyc")
input_hashes = {p: sha256_file(p) for p in sorted(set(protected_paths))}
print("Research branch:", BRANCH)
print("Python:", sys.version.split()[0], "| pandas:", pd.__version__)
print("Protected existing files:", len(input_hashes))

Starting notebook 09 — rental forecasting baselines...
Research branch: feat/rental-forecast-baselines
Python: 3.12.14 | pandas: 3.0.3
Protected existing files: 26


## 2. Verify the rental panel, without analysing the reserved outcomes

The CSV must match **your local quality report**. We verify schema, dates, complete quarterly coverage, source metadata, and positive numeric values. This is a structural check, not final-holdout scoring.

Only rental history will be used. The published count is retained in the original file but is **not** a forecast feature or an evaluation weight.

In [2]:
require(quality_report.get("source", {}).get("sha256") == EXPECTED_SOURCE_SHA256,
        "Rental quality report belongs to another workbook.")
require(quality_report.get("output_sha256", {}).get(INPUT.name) == input_hashes[INPUT],
        "Rental CSV differs from its pipeline report. Investigate; do not edit the hash.")
require(quality_report.get("csv_readback_matches") is True,
        "The rental pipeline has not recorded a successful CSV read-back.")
q = quality_report.get("quality", {})
require(q.get("rows") == 3286 and q.get("lga_count") == 31 and q.get("source_values_verified") == 6572,
        "Unexpected rental quality checkpoint.")
panel = read_panel_csv(INPUT)
require(panel.columns.tolist() == COLUMN_ORDER, "The 14-column rental schema changed.")
require(len(panel) == 3286 and panel.rental_area_name.nunique() == 31, "Unexpected panel size.")
require(not panel.duplicated(["rental_area_name", "property_type", "quarter"]).any(), "Duplicate rental key.")
for column, expected in {
    "property_type": "2 bedroom flats", "area_type": "LGA", "source_file": spec["filename"],
    "source_sheet": spec["sheet"], "source_reporting_quarter": "2025Q3",
}.items():
    require(panel[column].eq(expected).all(), f"Unexpected source scope: {column}")
for column in ["median_weekly_rent", "published_count"]:
    require(panel[column].notna().all() and np.isfinite(panel[column].astype(float)).all()
            and panel[column].gt(0).all(), f"Invalid numeric input: {column}")
periods = pd.PeriodIndex(panel.quarter, freq="Q-DEC")
expected_dates = periods.to_timestamp(how="end").normalize().as_unit("ns")
actual_dates = pd.DatetimeIndex(panel.quarter_end).as_unit("ns")
require(np.array_equal(actual_dates.asi8, expected_dates.asi8), "Quarter labels and dates disagree.")
ALL_QUARTERS = pd.period_range("1999Q2", "2025Q3", freq="Q-DEC")
for _, group in panel.groupby(["rental_area_name", "property_type"], sort=False):
    require(sorted(group.quarter.tolist()) == ALL_QUARTERS.astype(str).tolist(), "Missing or altered quarters.")
require(panel.source_median_cell.str.fullmatch(r"[A-Z]+[1-9][0-9]*").all(), "Missing source cell references.")
print("RENTAL INPUT CHECKPOINT PASSED")
print("Source rows: 3286 | Columns: 14 | LGAs: 31 | Quarters: 106")
print("Source CSV matches its local pipeline fingerprint: True")
print("Transport data loaded: False | Holdout forecasts scored: 0")

RENTAL INPUT CHECKPOINT PASSED
Source rows: 3286 | Columns: 14 | LGAs: 31 | Quarters: 106
Source CSV matches its local pipeline fingerprint: True
Transport data loaded: False | Holdout forecasts scored: 0


## 3. Fix the chronological protocol

The final eight quarters stay out of this experiment's predictions and metrics. We retain only keys to describe their reserved coverage, then remove their target values from the modelling frame.

The initial 79-quarter block is available history, **not 79 training errors to mix into validation**. These two baselines have no fitted coefficients. Later models will need proper training at each origin.

For target **2019Q1**, origin **2018Q4** is the latest allowed observation. Last observation copies **2018Q4**; seasonal naive copies **2018Q1**. For target **2019Q2**, the now-observed **2019Q1** may be used. This is rolling one-step evaluation, not a single 19-quarter forecast made in 2018.

In [3]:
EXPECTED_PLAN = {
    "protocol_id": "homereach-rental-baselines-v1", "target": "median_weekly_rent",
    "first_quarter": "1999Q2", "last_quarter": "2025Q3", "initial_history_end": "2018Q4",
    "validation_start": "2019Q1", "validation_end": "2023Q3",
    "holdout_start": "2023Q4", "holdout_end": "2025Q3",
    "forecast_horizon_quarters": 1, "seasonal_period_quarters": 4,
    "models": ["last_observation", "seasonal_naive"],
}
for key, expected in EXPECTED_PLAN.items():
    require(protocol.get(key) == expected, f"Unreviewed protocol change: {key}. Use a new reviewed experiment, not a silent split change.")
PLAN_SHA256 = hashlib.sha256(json.dumps(protocol, sort_keys=True, separators=(",", ":")).encode()).hexdigest()
INITIAL_END = pd.Period(protocol["initial_history_end"], freq="Q-DEC")
VALIDATION_QUARTERS = pd.period_range(protocol["validation_start"], protocol["validation_end"], freq="Q-DEC")
HOLDOUT_QUARTERS = pd.period_range(protocol["holdout_start"], protocol["holdout_end"], freq="Q-DEC")
require(INITIAL_END + 1 == VALIDATION_QUARTERS[0]
        and VALIDATION_QUARTERS[-1] + 1 == HOLDOUT_QUARTERS[0], "The splits must be consecutive and disjoint.")

MODEL_COLUMNS = ["rental_area_name", "property_type", "quarter", "median_weekly_rent",
                 "source_file", "source_sheet", "source_median_cell"]

def development_only(frame, validation_end):
    require(set(MODEL_COLUMNS).issubset(frame.columns), "Missing modelling columns.")
    index = pd.PeriodIndex(frame["quarter"], freq="Q-DEC")
    return frame.loc[index <= validation_end, MODEL_COLUMNS].copy()

initial_n = int((periods <= INITIAL_END).sum())
validation_n = int(periods.isin(VALIDATION_QUARTERS).sum())
holdout_keys = panel.loc[periods.isin(HOLDOUT_QUARTERS), ["rental_area_name", "property_type", "quarter"]].copy()
require((initial_n, validation_n, len(holdout_keys)) == (2449, 589, 248), "Unexpected chronological coverage.")
split_plan = pd.DataFrame([
    {"split": "initial_history", "first_quarter": "1999Q2", "last_quarter": "2018Q4", "quarters_per_lga": 79, "rows": initial_n, "scored_here": "no"},
    {"split": "rolling_validation", "first_quarter": "2019Q1", "last_quarter": "2023Q3", "quarters_per_lga": 19, "rows": validation_n, "scored_here": "yes"},
    {"split": "reserved_final_evaluation", "first_quarter": "2023Q4", "last_quarter": "2025Q3", "quarters_per_lga": 8, "rows": len(holdout_keys), "scored_here": "no"},
])
development = development_only(panel, VALIDATION_QUARTERS[-1])
LGAS = sorted(development.rental_area_name.unique())
del panel  # No holdout target values remain in the modelling frame.
OUT = ROOT / "data/interim/rental_forecast_baselines"
previous_manifest = OUT / "baseline_manifest.json"
if previous_manifest.is_file():
    previous = read_json(previous_manifest)
    require(previous.get("protocol_sha256") == PLAN_SHA256, "An existing run uses another protocol. Preserve it; do not silently replace the experiment.")
    require(previous.get("input_csv_sha256") == input_hashes[INPUT], "Existing results use a different rental input.")
display(split_plan)
print("Protocol:", protocol["protocol_id"])
print("Validation targets per model:", validation_n)
print("Reserved final-evaluation targets:", len(holdout_keys), "— not forecast or scored here")

,split,first_quarter,last_quarter,quarters_per_lga,rows,scored_here
0,initial_history,1999Q2,2018Q4,79,2449,no
1,rolling_validation,2019Q1,2023Q3,19,589,yes
2,reserved_final_evaluation,2023Q4,2025Q3,8,248,no


Protocol: homereach-rental-baselines-v1
Validation targets per model: 589
Reserved final-evaluation targets: 248 — not forecast or scored here


## 4. Predict using history only

For a target quarter `t`:

- `last_observation` uses the same area's value at `t − 1`.
- `seasonal_naive` uses its value at `t − 4`.

The low-level predictor accepts **history only** and rejects any target-quarter/future row. We use explicit quarter labels, not row positions, to find the seasonal match. Duplicate keys or gaps cause errors rather than silent shifts across missing quarters.

Each prediction carries a source quarter and workbook cell. Actual validation outcomes are attached **after** predictions have been constructed.

In [4]:
MODELS = ("last_observation", "seasonal_naive")
MODEL_LABELS = {"last_observation": "Last observation", "seasonal_naive": "Seasonal naive"}
SERIES_KEYS = ["rental_area_name", "property_type"]
TARGET_KEYS = SERIES_KEYS + ["target_quarter"]


def checked_series(group):
    require(set(MODEL_COLUMNS).issubset(group.columns), "Missing model-history fields.")
    require(len(group) > 0 and group[SERIES_KEYS].drop_duplicates().shape[0] == 1,
            "A history must contain exactly one LGA/property series.")
    require(not group.quarter.duplicated().any(), "Duplicate quarter in a series.")
    ordered = group.assign(_period=pd.PeriodIndex(group.quarter, freq="Q-DEC")).sort_values("_period", kind="stable").reset_index(drop=True)
    index = pd.PeriodIndex(ordered._period, freq="Q-DEC")
    require(index.equals(pd.period_range(index.min(), index.max(), freq="Q-DEC")), "Gap in quarterly history.")
    values = pd.to_numeric(ordered.median_weekly_rent, errors="raise").astype(float)
    require(np.isfinite(values).all() and values.gt(0).all(), "Rents must be positive finite values.")
    return ordered


def predict_from_history(history, target_quarter, model):
    require(model in MODELS, "Unknown baseline model.")
    target = pd.Period(target_quarter, freq="Q-DEC")
    ordered = checked_series(history)
    index = pd.PeriodIndex(ordered._period, freq="Q-DEC")
    require((index < target).all(), "Target-quarter or future rows must not reach the predictor.")
    require(index.max() == target - 1, "Latest observed quarter must be the forecast origin t-1.")
    source_quarter = target - (1 if model == "last_observation" else 4)
    match = ordered.loc[ordered._period.eq(source_quarter)]
    require(len(match) == 1, "Required earlier quarter is unavailable.")
    source = match.iloc[0]
    return {
        "model": model, "rental_area_name": str(source.rental_area_name), "property_type": str(source.property_type),
        "forecast_origin_quarter": str(target - 1), "target_quarter": str(target), "horizon_quarters": 1,
        "forecast_source_quarter": str(source_quarter), "available_history_rows": len(ordered),
        "forecast_weekly_rent_aud": float(source.median_weekly_rent),
        "source_file": str(source.source_file), "source_sheet": str(source.source_sheet),
        "forecast_source_median_cell": str(source.source_median_cell),
    }


def rolling_predictions(frame, targets):
    targets = pd.PeriodIndex(targets, freq="Q-DEC")
    require(len(targets) > 0 and not targets.has_duplicates and targets.is_monotonic_increasing,
            "Targets must be unique and increasing.")
    require(pd.PeriodIndex(frame.quarter, freq="Q-DEC").max() <= targets.max(),
            "Remove later/holdout rows before constructing validation forecasts.")
    result = []
    for _, group in frame.groupby(SERIES_KEYS, sort=True, observed=True):
        ordered = checked_series(group)
        require(targets.isin(pd.PeriodIndex(ordered._period)).all(), "A validation target is missing.")
        for target in targets:
            history = ordered.loc[ordered._period.lt(target), MODEL_COLUMNS].copy()
            for model in MODELS:
                result.append(predict_from_history(history, target, model))
    return pd.DataFrame(result).sort_values(["model", *SERIES_KEYS, "target_quarter"], kind="stable").reset_index(drop=True)


def attach_outcomes(predictions, development_frame, targets):
    target_labels = pd.PeriodIndex(targets, freq="Q-DEC").astype(str).tolist()
    require(predictions.target_quarter.isin(target_labels).all(), "A prediction is outside validation.")
    actual = development_frame.loc[development_frame.quarter.isin(target_labels),
                                  SERIES_KEYS + ["quarter", "median_weekly_rent", "source_median_cell"]].rename(columns={
        "quarter": "target_quarter", "median_weekly_rent": "actual_weekly_rent_aud", "source_median_cell": "target_source_median_cell"})
    require(not actual.duplicated(TARGET_KEYS).any(), "Duplicate target outcome.")
    scored = predictions.merge(actual, on=TARGET_KEYS, how="left", validate="many_to_one", indicator=True)
    require(scored._merge.eq("both").all(), "Missing actual outcome for a prediction.")
    scored = scored.drop(columns="_merge")
    require(np.isfinite(scored.actual_weekly_rent_aud.astype(float)).all(), "Invalid actual outcome.")
    scored["error_aud_week"] = scored.forecast_weekly_rent_aud - scored.actual_weekly_rent_aud
    scored["absolute_error_aud_week"] = scored.error_aud_week.abs()
    scored["squared_error"] = scored.error_aud_week.pow(2)
    return scored


def metrics(frame, group_columns):
    require(len(frame) > 0 and np.isfinite(frame.error_aud_week).all(), "Metrics need finite, nonempty forecast errors.")
    rows = []
    grouper = group_columns[0] if len(group_columns) == 1 else group_columns
    for key, group in frame.groupby(grouper, sort=True, observed=True):
        keys = (key,) if len(group_columns) == 1 else key
        errors = group.error_aud_week.to_numpy(dtype=float)
        rows.append({**dict(zip(group_columns, keys)), "n_forecasts": len(errors),
                     "mae_aud_week": float(np.mean(np.abs(errors))),
                     "rmse_aud_week": float(np.sqrt(np.mean(errors ** 2))),
                     "mean_signed_error_aud_week": float(np.mean(errors))})
    return pd.DataFrame(rows)

print("History-only forecasting and scoring functions defined.")

History-only forecasting and scoring functions defined.


## 5. Small examples test the method before the real predictions

These synthetic tests cover exact expected answers, shuffled rows, duplicate/gapped histories, missing targets, cross-LGA isolation, future-value contamination, and error arithmetic. The current target can change its eventual score but must not change the prediction already made for it.

They are **not additional pytest cases**; your existing 352-test suite is unchanged.

In [5]:
def expect_value_error(call, phrase):
    try:
        call()
    except ValueError as exc:
        require(phrase in str(exc), f"Wrong rejection reason: {exc}")
    else:
        raise AssertionError(f"Expected rejection containing: {phrase}")


def toy_series(name="Fixture A", offset=0.0):
    quarters = pd.period_range("2017Q1", "2020Q4", freq="Q-DEC")
    return pd.DataFrame({
        "rental_area_name": name, "property_type": "2 bedroom flats", "quarter": quarters.astype(str),
        "median_weekly_rent": [100.0 + i*i + offset for i in range(len(quarters))],
        "source_file": "fictional.xlsx", "source_sheet": "fixture", "source_median_cell": [f"A{i+1}" for i in range(len(quarters))],
    })

checks = []
toy = toy_series()
toy_targets = pd.period_range("2019Q1", "2019Q4", freq="Q-DEC")
toy_dev = development_only(toy, toy_targets[-1])
history = toy_dev.iloc[:8].copy()
last = predict_from_history(history, "2019Q1", "last_observation")
seasonal = predict_from_history(history, "2019Q1", "seasonal_naive")
require(last["forecast_weekly_rent_aud"] == 149 and last["forecast_source_quarter"] == "2018Q4", "Wrong last-observation answer.")
require(seasonal["forecast_weekly_rent_aud"] == 116 and seasonal["forecast_source_quarter"] == "2018Q1", "Wrong seasonal answer.")
checks.append("known answers and source quarters")
base = rolling_predictions(toy_dev, toy_targets)
pd.testing.assert_frame_equal(base, rolling_predictions(toy_dev.sample(frac=1, random_state=7), toy_targets))
checks.append("row-order invariance")
expect_value_error(lambda: predict_from_history(toy_dev.iloc[:9], "2019Q1", "last_observation"), "future rows")
checks.append("current/future row rejection")
mutated = toy.copy()
mutated.loc[pd.PeriodIndex(mutated.quarter, freq="Q-DEC") > toy_targets[-1], "median_weekly_rent"] = 999999.0
pd.testing.assert_frame_equal(base, rolling_predictions(development_only(mutated, toy_targets[-1]), toy_targets))
checks.append("holdout-value mutation invariance")
mutated = toy_dev.copy()
mutated.loc[mutated.quarter.eq("2019Q1"), "median_weekly_rent"] = 9999.0
changed = rolling_predictions(mutated, toy_targets)
pd.testing.assert_frame_equal(base.loc[base.target_quarter.eq("2019Q1")].reset_index(drop=True),
                              changed.loc[changed.target_quarter.eq("2019Q1")].reset_index(drop=True))
checks.append("target value cannot alter its own prediction")
next_last = changed.loc[changed.model.eq("last_observation") & changed.target_quarter.eq("2019Q2")]
require(next_last.forecast_weekly_rent_aud.iloc[0] == 9999, "An observed validation result should enter the next origin.")
checks.append("rolling one-step observation update")
expect_value_error(lambda: rolling_predictions(toy_dev.drop(index=2), toy_targets), "Gap")
checks.append("gap rejection")
expect_value_error(lambda: rolling_predictions(pd.concat([toy_dev, toy_dev.iloc[[0]]]), toy_targets), "Duplicate")
checks.append("duplicate rejection")
bad = toy_dev.copy(); bad.loc[0, "median_weekly_rent"] = np.inf
expect_value_error(lambda: rolling_predictions(bad, toy_targets), "positive finite")
checks.append("nonfinite target rejection")
m = metrics(pd.DataFrame({"model": ["fixture", "fixture"], "error_aud_week": [10.0, -20.0]}), ["model"]).iloc[0]
require(m.mae_aud_week == 15 and np.isclose(m.rmse_aud_week, math.sqrt(250)) and m.mean_signed_error_aud_week == -5,
        "Metric formula check failed.")
checks.append("MAE RMSE and signed-error arithmetic")
combined = pd.concat([toy_dev, development_only(toy_series("Fixture B", 1000), toy_targets[-1])], ignore_index=True)
combined_predictions = rolling_predictions(combined, toy_targets)
pd.testing.assert_frame_equal(base, combined_predictions.loc[combined_predictions.rental_area_name.eq("Fixture A")].reset_index(drop=True))
checks.append("cross-LGA isolation")
expect_value_error(lambda: rolling_predictions(toy_dev.iloc[:-1], toy_targets), "target is missing")
checks.append("missing validation target rejection")
require(len(checks) == 12, "Unexpected self-check count.")
print("Twelve baseline self-check groups passed.")

Twelve baseline self-check groups passed.


## 6. Produce validation forecasts and independently verify each historical match

There are **31 × 19 = 589 targets per model**, hence **1,178 prediction rows**. Source observations may be earlier validation outcomes, but are always earlier than their own target.

The independent check uses grouped lags on the complete **development-only** history after verifying no gaps. It compares the forecast **value, source quarter and source cell**, not just the number of predictions.

In [6]:
forecasts = rolling_predictions(development, VALIDATION_QUARTERS)
require(len(forecasts) == 1178, "Expected 589 predictions for each of two methods.")
require(forecasts.groupby("model").size().to_dict() == {m: 589 for m in MODELS}, "Unbalanced model coverage.")
require(not forecasts.duplicated(["model", *TARGET_KEYS]).any(), "Duplicate prediction key.")
source_periods = pd.PeriodIndex(forecasts.forecast_source_quarter, freq="Q-DEC")
target_periods = pd.PeriodIndex(forecasts.target_quarter, freq="Q-DEC")
origin_periods = pd.PeriodIndex(forecasts.forecast_origin_quarter, freq="Q-DEC")
require((source_periods <= origin_periods).all() and (origin_periods + 1 == target_periods).all(), "Temporal leakage in forecast lineage.")
require(not target_periods.isin(HOLDOUT_QUARTERS).any(), "Holdout targets reached validation predictions.")
reference = development.sort_values(SERIES_KEYS + ["quarter"], kind="stable").copy()
independent_rows = 0
for model, lag in [("last_observation", 1), ("seasonal_naive", 4)]:
    grouped = reference.groupby(SERIES_KEYS, sort=False, observed=True)
    expected = reference[SERIES_KEYS + ["quarter"]].rename(columns={"quarter": "target_quarter"}).copy()
    expected["expected_forecast"] = grouped.median_weekly_rent.shift(lag)
    expected["expected_source_quarter"] = grouped.quarter.shift(lag)
    expected["expected_source_cell"] = grouped.source_median_cell.shift(lag)
    compared = forecasts.loc[forecasts.model.eq(model)].merge(expected, on=TARGET_KEYS, how="left", validate="one_to_one")
    require(compared.expected_forecast.notna().all(), "An independent lag is missing.")
    require(np.array_equal(compared.forecast_weekly_rent_aud.to_numpy(), compared.expected_forecast.to_numpy()), "Forecast differs from independent lag.")
    require(compared.forecast_source_quarter.eq(compared.expected_source_quarter).all()
            and compared.forecast_source_median_cell.eq(compared.expected_source_cell).all(), "Source lineage differs from independent lag.")
    independent_rows += len(compared)
scored = attach_outcomes(forecasts, development, VALIDATION_QUARTERS)
require(len(scored) == independent_rows == 1178, "Prediction/outcome rows changed.")
print("VALIDATION FORECAST CHECKPOINT PASSED")
print("Predictions:", len(scored), "| Independent lag/value/cell matches:", independent_rows)
print("Source quarters after forecast origin: 0 | Final-holdout predictions: 0")
display(scored.loc[scored.rental_area_name.eq("Melbourne"), [
    "model", "forecast_origin_quarter", "target_quarter", "forecast_source_quarter",
    "forecast_weekly_rent_aud", "actual_weekly_rent_aud", "forecast_source_median_cell", "target_source_median_cell",
]].head(6))

VALIDATION FORECAST CHECKPOINT PASSED
Predictions: 1178 | Independent lag/value/cell matches: 1178
Source quarters after forecast origin: 0 | Final-holdout predictions: 0


,model,forecast_origin_quarter,target_quarter,forecast_source_quarter,forecast_weekly_rent_aud,actual_weekly_rent_aud,forecast_source_median_cell,target_source_median_cell
323,last_observation,2018Q4,2019Q1,2018Q4,580.0,600.0,FD63,FF63
324,last_observation,2019Q1,2019Q2,2019Q1,600.0,600.0,FF63,FH63
325,last_observation,2019Q2,2019Q3,2019Q2,600.0,600.0,FH63,FJ63
326,last_observation,2019Q3,2019Q4,2019Q3,600.0,585.0,FJ63,FL63
327,last_observation,2019Q4,2020Q1,2019Q4,585.0,600.0,FL63,FN63
328,last_observation,2020Q1,2020Q2,2020Q1,600.0,500.0,FN63,FP63


## 7. Measure forecast errors — only on validation

Our signed error is **forecast − actual**: positive means over-prediction; negative means under-prediction. (Some references use the opposite sign; the absolute and squared metrics are unchanged.)

- **MAE** = mean absolute error. An MAE of $15 means an average absolute error of $15 in the **weekly rent statistic**, not $15 per quarter and not 15%.
- **RMSE** = square root of the mean squared error. It is also in AUD/week and penalises large misses more strongly.
- **Mean signed error** summarises direction, not accuracy: positive and negative errors can cancel.

Each LGA-quarter gets equal weight. These are area-statistic forecasts; we do not weight errors by published letting counts. With equal 19-quarter coverage, pooled MAE equals the mean of LGA MAEs. **Pooled RMSE is not generally the average of LGA RMSEs.** No confidence intervals or significance claims are produced from these dependent historical errors.

In [7]:
overall_metrics = metrics(scored, ["model"])
lga_metrics = metrics(scored, ["model", "rental_area_name", "property_type"])
quarter_metrics = metrics(scored, ["model", "target_quarter"])
require(overall_metrics.n_forecasts.eq(589).all(), "Incorrect pooled denominator.")
require(len(lga_metrics) == 62 and lga_metrics.n_forecasts.eq(19).all(), "Incorrect LGA denominator.")
require(len(quarter_metrics) == 38 and quarter_metrics.n_forecasts.eq(31).all(), "Incorrect quarter denominator.")
for model in MODELS:
    summary = overall_metrics.loc[overall_metrics.model.eq(model)].iloc[0]
    by_lga = lga_metrics.loc[lga_metrics.model.eq(model)]
    require(np.isclose(summary.mae_aud_week, by_lga.mae_aud_week.mean()), "Pooled/macro MAE reconciliation failed.")
    require(np.isclose(summary.rmse_aud_week, np.sqrt(np.average(by_lga.rmse_aud_week**2, weights=by_lga.n_forecasts))),
            "Pooled RMSE reconciliation failed.")
model_comparison = lga_metrics.pivot(index=["rental_area_name", "property_type"], columns="model", values="mae_aud_week").reset_index()
model_comparison.columns.name = None
model_comparison = model_comparison.rename(columns={"last_observation": "last_observation_mae_aud_week", "seasonal_naive": "seasonal_naive_mae_aud_week"})
model_comparison["seasonal_minus_last_mae_aud_week"] = model_comparison.seasonal_naive_mae_aud_week - model_comparison.last_observation_mae_aud_week
model_comparison["lower_validation_mae"] = np.where(np.isclose(model_comparison.seasonal_minus_last_mae_aud_week, 0), "tie",
    np.where(model_comparison.seasonal_minus_last_mae_aud_week.gt(0), "last_observation", "seasonal_naive"))
display(overall_metrics.round(4))
print("Lower validation MAE by LGA (descriptive only):")
display(model_comparison.lower_validation_mae.value_counts().rename_axis("method").reset_index(name="lga_count"))
print("No final-period scores, published forecasts, model winner for deployment, or current-rent estimates have been produced.")

,model,n_forecasts,mae_aud_week,rmse_aud_week,mean_signed_error_aud_week
0,last_observation,589,10.9728,16.2303,-4.3582
1,seasonal_naive,589,25.2937,38.9464,-13.0934


Lower validation MAE by LGA (descriptive only):


,method,lga_count
0,last_observation,29
1,seasonal_naive,2


No final-period scores, published forecasts, model winner for deployment, or current-rent estimates have been produced.


## 8. Chart 1 — Overall validation error

Lower bars mean smaller average absolute errors across the 589 LGA-quarter targets for that method. The hover also shows RMSE and signed error. This is a benchmark result on the stated validation interval, not a final-test result or accuracy guarantee.

In [8]:
figures = {}
figures["01_validation_mae.html"] = px.bar(
    overall_metrics.assign(method=overall_metrics.model.map(MODEL_LABELS)), x="method", y="mae_aud_week",
    hover_data={"n_forecasts": True, "rmse_aud_week": ":.2f", "mean_signed_error_aud_week": ":.2f"},
    labels={"method": "Forecast method", "mae_aud_week": "Validation MAE (AUD/week)"},
    title="One-quarter-ahead baselines · validation 2019Q1–2023Q3",
)
figures["01_validation_mae.html"].update_layout(height=440)
figures["01_validation_mae.html"].show()

## 9. Charts 2 and 3 — Errors vary across quarters and LGAs

The quarterly chart averages over the same 31 LGAs at every target. A spike shows a harder-to-predict quarter for that baseline; the plot alone does not identify a cause.

The LGA chart gives equal 19-quarter evaluation coverage to every area. Areas are alphabetical rather than ordered as better or worse places to live. Lower forecast error is not a recommendation about an area.

In [9]:
quarter_plot = quarter_metrics.assign(method=quarter_metrics.model.map(MODEL_LABELS)).sort_values(["method", "target_quarter"])
figures["02_validation_mae_by_quarter.html"] = px.line(
    quarter_plot, x="target_quarter", y="mae_aud_week", color="method", markers=True,
    labels={"target_quarter": "Target quarter (validation only)", "mae_aud_week": "MAE across 31 LGAs (AUD/week)", "method": "Method"},
    title="Validation error over time · rolling one-quarter forecasts",
)
figures["02_validation_mae_by_quarter.html"].update_layout(height=460, legend=dict(orientation="h", y=1.12))
figures["02_validation_mae_by_quarter.html"].show()
lga_plot = lga_metrics.assign(method=lga_metrics.model.map(MODEL_LABELS))
figures["03_validation_mae_by_lga.html"] = px.bar(
    lga_plot, y="rental_area_name", x="mae_aud_week", color="method", barmode="group", orientation="h",
    category_orders={"rental_area_name": LGAS},
    labels={"rental_area_name": "Rental-source LGA", "mae_aud_week": "Validation MAE (AUD/week)", "method": "Method"},
    title="Per-LGA validation error · same 19 targets for each area",
)
figures["03_validation_mae_by_lga.html"].update_layout(height=1040, legend=dict(orientation="h", y=1.04), margin=dict(l=170))
figures["03_validation_mae_by_lga.html"].show()

## 10. Chart 4 — A forecast trace you can explain

Melbourne is the same illustrative LGA used in earlier audits, chosen before looking at forecast errors. Each marker is a separate **one-quarter-ahead** forecast, not a multi-quarter path issued at the start of validation.

Actual outcomes are shown only after predictions have been constructed. The chart ends at **2023Q3**. No reserved final-period observations are plotted. Nominal medians describe changing sets of lettings, not one fixed dwelling.

In [10]:
EXAMPLE_LGA = "Melbourne"
example = scored.loc[scored.rental_area_name.eq(EXAMPLE_LGA)].sort_values(["model", "target_quarter"])
require(len(example) == 38, "Expected 19 validation forecasts per model for the example.")
actual_example = example[TARGET_KEYS + ["actual_weekly_rent_aud"]].drop_duplicates().sort_values("target_quarter")
fig = go.Figure()
fig.add_trace(go.Scatter(x=actual_example.target_quarter, y=actual_example.actual_weekly_rent_aud, mode="lines+markers", name="Published actual"))
for model in MODELS:
    data = example.loc[example.model.eq(model)]
    fig.add_trace(go.Scatter(
        x=data.target_quarter, y=data.forecast_weekly_rent_aud, mode="lines+markers", name=MODEL_LABELS[model],
        customdata=data[["forecast_origin_quarter", "forecast_source_quarter", "forecast_source_median_cell"]].to_numpy(),
        hovertemplate="Target: %{x}<br>Forecast: $%{y:.0f}/week<br>Origin: %{customdata[0]}<br>Copied quarter: %{customdata[1]}<br>Workbook cell: %{customdata[2]}<extra>%{fullData.name}</extra>",
    ))
fig.update_layout(title="Melbourne · rolling forecasts against published medians (validation only)",
                  xaxis_title="Target quarter", yaxis_title="Nominal median weekly rent (AUD/week)",
                  height=500, legend=dict(orientation="h", y=1.12))
figures["04_melbourne_validation_forecasts.html"] = fig
fig.show()

## 11. Save validation evidence and verify it after reloading

This writes six CSV tables, four self-contained HTML charts, a copy of the fixed protocol, a result manifest, and a generated findings report. Only this experiment's output directory and report are written.

Prediction/source dates and integer counts are checked exactly. Small floating-point serialization differences in decimal metrics are allowed only within `1e-12` relative/absolute tolerance. The report is published after the tables have been verified; avoid simultaneous writers. No holdout predictions, raw GTFS, fitted model or live-app data is exported.

A rerun replaces this notebook's own generated report; keep personal notes in another file. Changed existing protocol/input fingerprints stop the run instead of silently replacing another experiment.

In [11]:
exports = {
    "split_plan.csv": split_plan,
    "validation_predictions.csv": scored,
    "validation_metrics_overall.csv": overall_metrics,
    "validation_metrics_by_lga.csv": lga_metrics,
    "validation_metrics_by_quarter.csv": quarter_metrics,
    "validation_model_comparison_by_lga.csv": model_comparison,
}

def schema_for(frame):
    result = {}
    for name, dtype in frame.dtypes.items():
        if pd.api.types.is_integer_dtype(dtype):
            result[name] = "int64"
        elif pd.api.types.is_float_dtype(dtype):
            result[name] = "float64"
        else:
            result[name] = "string"
    return result


def compare_readback(expected, actual):
    require(expected.columns.tolist() == actual.columns.tolist() and len(expected) == len(actual), "CSV shape/schema changed.")
    for column, kind in schema_for(expected).items():
        if kind == "float64":
            np.testing.assert_allclose(expected[column].astype(float), actual[column].astype(float), rtol=1e-12, atol=1e-12, equal_nan=False)
        elif kind == "int64":
            require(np.array_equal(expected[column].to_numpy(), actual[column].to_numpy()), f"Integer round-trip mismatch: {column}")
        else:
            require(expected[column].astype("string").reset_index(drop=True).equals(actual[column].astype("string").reset_index(drop=True)), f"Text round-trip mismatch: {column}")

# Final boundary checks before any research outputs are written.
require(not pd.PeriodIndex(scored.target_quarter, freq="Q-DEC").isin(HOLDOUT_QUARTERS).any(), "Do not export holdout forecasts here.")
for path, digest in input_hashes.items():
    require(sha256_file(path) == digest, f"Protected input changed during computation: {path.relative_to(ROOT)}")
OUT.mkdir(parents=True, exist_ok=True)
output_schemas = {}
with TemporaryDirectory(prefix=".baseline-stage-", dir=OUT) as temp_dir:
    stage = Path(temp_dir)
    for filename, table in exports.items():
        destination = stage / filename
        table.to_csv(destination, index=False, encoding="utf-8", lineterminator="\n")
        schema = schema_for(table); output_schemas[filename] = schema
        reloaded = pd.read_csv(destination, dtype=schema, keep_default_na=False)
        compare_readback(table, reloaded)
    for filename in exports:
        (stage / filename).replace(OUT / filename)
figure_dir = OUT / "figures"; figure_dir.mkdir(exist_ok=True)
for filename, figure in figures.items():
    figure.write_html(figure_dir / filename, include_plotlyjs=True, full_html=True, config={"responsive": True, "displaylogo": False})
(OUT / "forecast_protocol.json").write_text(json.dumps(protocol, indent=2) + "\n", encoding="utf-8")
print("Six CSV tables reloaded and matched; four interactive charts saved.")

Six CSV tables reloaded and matched; four interactive charts saved.


## 12. Record the baseline findings and final checkpoint

A validation advantage is a benchmark for the next experiment, not proof of generalisation or a reason to modify the frozen application. Keep both baseline rows when comparing future models. A more complex model must be tested using the same target dates and information rules.

The signed-error sign convention, reserved period, workbook-vintage limitation and previous EDA exposure are included in the report and manifest.

In [12]:
REPORT_PATH = ROOT / "docs/rental_forecast_baselines.md"
metric_lines = ["| Method | Validation targets | MAE (AUD/week) | RMSE (AUD/week) | Mean signed error (forecast − actual) |",
                "|---|---:|---:|---:|---:|"]
for row in overall_metrics.itertuples(index=False):
    metric_lines.append(f"| {MODEL_LABELS[row.model]} | {row.n_forecasts} | {row.mae_aud_week:.4f} | {row.rmse_aud_week:.4f} | {row.mean_signed_error_aud_week:.4f} |")
created_at = datetime.now(timezone.utc).isoformat()
report_lines = [
    "# HomeReach Melbourne — Rental forecast baselines", "", f"Generated (UTC): {created_at}", "",
    "## Target and fixed protocol", "",
    "Retrospective one-quarter-ahead forecasts of published nominal median weekly rent for two-bedroom flats, separately for 31 rental-source metropolitan LGAs.",
    "No transport, current-quarter published counts, or future rental values are predictors. Both baselines have horizon one quarter.",
    "Initial history: 1999Q2–2018Q4 (79 quarters). Rolling validation: 2019Q1–2023Q3 (19 targets per LGA).",
    "Reserved final evaluation: 2023Q4–2025Q3 (8 quarters; 248 LGA-quarter outcomes). No predictions or error scores for that period are created here.",
    "These dates were specified in config/rental_forecast_protocol.json before error calculation; they are project choices, not official publisher labels.", "",
    "## Validation results", "", *metric_lines, "",
    "MAE is the primary metric. Every LGA-quarter has equal weight; errors are not weighted by published counts. A positive signed error is over-prediction.",
    "Pooled RMSE is calculated from all squared errors, not by averaging per-area RMSEs. These correlated historical errors have no claimed confidence interval or significance test.", "",
    "## Method", "",
    "For target t, last observation copies t−1 and seasonal naive copies t−4 for the same area/category. Origin is t−1 for both.",
    "At each later validation origin, previously observed validation outcomes enter the available history. This is repeated one-step evaluation, not a fixed-origin multi-step forecast.",
    "Predictions are constructed before actual outcomes are attached; source quarter and original workbook cell are retained for every forecast.", "",
    "## Verification", "",
    f"- Rental input: {INPUT.relative_to(ROOT)}; SHA-256 {input_hashes[INPUT]}.",
    f"- Original source workbook SHA-256: {EXPECTED_SOURCE_SHA256}.",
    f"- Protocol SHA-256: {PLAN_SHA256}.",
    "- 589 validation forecasts per method; 1,178 prediction rows and independent value/quarter/cell matches.",
    "- Twelve synthetic self-check groups passed; these are notebook checks, not new pytest cases.",
    "- Pooled, per-LGA and per-quarter counts reconciled; six CSV tables reloaded and matched.",
    "- Four interactive charts contain validation data only; 0 final-holdout forecasts and 0 current/future published forecasts.",
    "- Protected inputs, application and frozen release were not changed.", "",
    "## Interpretation limits", "",
    "Only one 2025Q3 workbook vintage is available. Historical release lags and revisions are not reconstructed; this is NOT a real-time backtest.",
    "The recent holdout outcomes have already appeared in prior descriptive EDA and the explorer. They are reserved from modelling evaluation here, not a pristine blind test.",
    "The full source CSV is read for structural checks; its holdout outcomes are excluded before model construction and from metrics/charts/prediction exports.",
    "Source-area labels follow the publisher's historical series; this does not independently harmonise historic administrative-boundary changes.",
    "Published medians describe changing sets of lettings; this is not an individual-property forecast, inflation-adjusted series, live-rent estimate or area recommendation.",
    "No out-of-sample uncertainty intervals, causal claims, fitted ML model or deployment recommendation are produced. Keep the final period reserved until model choices are fixed.", "",
    "## Reproduction", "", "Run notebooks/09_rental_forecast_baselines.ipynb on feat/rental-forecast-baselines with the existing homereach kernel.",
    "Research outputs are under data/interim/rental_forecast_baselines; this report is regenerated on rerun. Keep personal notes separately.", "",
    "## Primary references", "",
    "- https://otexts.com/fpp3/simple-methods.html", "- https://otexts.com/fpp3/tscv.html", "- https://otexts.com/fpp3/accuracy.html",
    "- https://www.dffh.vic.gov.au/publications/rental-report", "",
]
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
REPORT_PATH.write_text("\n".join(report_lines), encoding="utf-8")
for path, digest in input_hashes.items():
    require(sha256_file(path) == digest, f"Protected file changed: {path.relative_to(ROOT)}")
manifest = {
    "created_at_utc": created_at, "builder": "notebooks/09_rental_forecast_baselines.ipynb", "research_branch": BRANCH,
    "versions": {"python": sys.version.split()[0], "pandas": pd.__version__, "numpy": np.__version__, "plotly": plotly.__version__},
    "source_workbook_sha256": EXPECTED_SOURCE_SHA256, "input_csv_sha256": input_hashes[INPUT],
    "protocol_sha256": PLAN_SHA256, "protocol": protocol,
    "quality": {"input_rows": 3286, "lga_count": 31, "initial_history_rows": initial_n,
                "validation_targets_per_model": validation_n, "validation_prediction_rows": len(scored),
                "reserved_holdout_rows": len(holdout_keys), "holdout_forecasts": 0, "holdout_scores": 0,
                "independent_lag_source_matches": independent_rows, "self_check_groups": len(checks),
                "csv_readback_matches": True, "csv_tables": len(exports), "charts": len(figures),
                "transport_features_used": False, "current_rent_forecast_produced": False, "protected_files_unchanged": True},
    "self_checks": checks, "validation_metrics": overall_metrics.to_dict(orient="records"),
    "output_schemas": output_schemas,
    "output_sha256": {str(p.relative_to(OUT)).replace("\\", "/"): sha256_file(p)
                      for p in sorted(OUT.rglob("*")) if p.is_file() and p.name != "baseline_manifest.json"},
    "report_sha256": sha256_file(REPORT_PATH),
}
(OUT / "baseline_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
print("BASELINE VALIDATION CHECKPOINT PASSED")
print("Rental LGAs: 31 | Source quarters: 106")
print("Initial history: 1999Q2–2018Q4 (79 quarters)")
print("Validation: 2019Q1–2023Q3 (19 quarters)")
print("Reserved final evaluation: 2023Q4–2025Q3 (8 quarters)")
print("Validation targets per model: 589")
print("Validation forecast rows: 1178")
print("Final-holdout forecasts scored: 0")
print("Independent historical value/quarter/cell matches: 1178")
print("Baseline self-check groups: 12 passed")
print("CSV tables saved and reloaded: 6")
print("Interactive charts saved: 4")
print("Transport features used: False | Current-rent forecast produced: False")
print("Protected inputs and deployed release unchanged: True")
print("Saved report:", REPORT_PATH.relative_to(ROOT))
print("Saved outputs:", OUT.relative_to(ROOT))

BASELINE VALIDATION CHECKPOINT PASSED
Rental LGAs: 31 | Source quarters: 106
Initial history: 1999Q2–2018Q4 (79 quarters)
Validation: 2019Q1–2023Q3 (19 quarters)
Reserved final evaluation: 2023Q4–2025Q3 (8 quarters)
Validation targets per model: 589
Validation forecast rows: 1178
Final-holdout forecasts scored: 0
Independent historical value/quarter/cell matches: 1178
Baseline self-check groups: 12 passed
CSV tables saved and reloaded: 6
Interactive charts saved: 4
Transport features used: False | Current-rent forecast produced: False
Protected inputs and deployed release unchanged: True
Saved report: docs\rental_forecast_baselines.md
Saved outputs: data\interim\rental_forecast_baselines


## Your checkpoint and interpretation exercise

Save the notebook with **Ctrl+S**. Send the final checkpoint and the two-row MAE/RMSE table, or the first error. Do not change a validation check, the protocol dates, or the source fingerprints merely to make execution pass. No new Git commit is required until this run succeeds.

You should be able to explain **why seasonal naive is still a one-quarter forecast**, why earlier validation observations may be used at later origins, and why a one-vintage retrospective evaluation is not evidence of real-time availability.

The next modelling step is a small rental-only challenger evaluated on these same validation targets. The reserved final evaluation period must stay unscored until the experiment's model choices and metrics are fixed. The live explorer stays descriptive during this research phase.